# Capstone — Prioritizing Content Refresh Reviews Using Pre-Decision Search Signals

This end-to-end notebook mirrors our deployed research paper ([`docs/index.html`](../../docs/index.html)) and Markdown report ([`work/capstone_report.md`](../capstone_report.md)). All numbers, tables, and charts are reproduced directly from the repository datasets with `RANDOM_STATE = 42`.

## 1. Question

**Core Research Question:** *"Using only information available before the decision moment, which content items should be reviewed first for a possible refresh, and what signals explain that priority?"*

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_repo_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        if (p / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return p
    return cur

REPO_ROOT = find_repo_root()
import json

with open(REPO_ROOT / "work" / "outputs" / "capstone_metrics.json", "r", encoding="utf-8") as f:
    M = json.load(f)

print("Project:", M["project"])
print("Lane   :", M["lane"])
print("Seed   :", M["random_state"])
print("Week-4 Baseline Formula:", M["w04_baseline_definition"]["formula"])


Project: FlyRank Machine Learning Week-8 Capstone
Lane   : Content Refresh Priority Prediction
Seed   : 42
Week-4 Baseline Formula: score = 2*(impressions >= 1000) + 2*(staleness_days >= 14) + 1*(position >= 8)


## 2. Data

- **Population A (`w05_ml_practice_dataset.csv`, $N = 100$):** 6 columns (`item_id, impressions, clicks, staleness_days, position, target`), positive base rate `0.6100` (`0.6000` in the 20-row stratified test split).
- **Population B (`data/raw/content_refresh_anonymized.csv`, $N = 30,000$):** 44 columns across `32` pseudonymized clients (`client_id`), positive declining base rate `0.5421` (`16,262 / 30,000`).
- **Exclusions:** `trend_direction`, `trend_pct`, `*_last_30d`, `*_prev_30d`, product rule scores (`health_score`, `priority_score`, `action_type`, `refresh_tier`), and identifiers (`content_id`, `client_id`, `item_id`) are strictly excluded from feature matrices.

In [2]:
data_overview = pd.DataFrame([
    {"Cohort": "W04 Practice Dataset", "File": "ml07_baseline_practice_dataset.csv", "Rows": 30, "Clients": 1, "Base_Rate": "Unlabeled queue"},
    {"Cohort": "W05/W06 Practice Dataset (Pop. A)", "File": "w05_ml_practice_dataset.csv", "Rows": M["w05_practice_evaluation"]["n_total"], "Clients": 1, "Base_Rate": M["w05_practice_evaluation"]["positive_base_rate_total"]},
    {"Cohort": "FlyRank Starter Dataset (Pop. B)", "File": "data/raw/content_refresh_anonymized.csv", "Rows": M["flyrank_30k_evaluation"]["n_total"], "Clients": M["flyrank_30k_evaluation"]["n_clients"], "Base_Rate": M["flyrank_30k_evaluation"]["positive_base_rate_total"]},
])
print(data_overview.to_string(index=False))


                           Cohort                                    File  Rows  Clients       Base_Rate
             W04 Practice Dataset      ml07_baseline_practice_dataset.csv    30        1 Unlabeled queue
W05/W06 Practice Dataset (Pop. A)             w05_ml_practice_dataset.csv   100        1            0.61
 FlyRank Starter Dataset (Pop. B) data/raw/content_refresh_anonymized.csv 30000       32          0.5421


## 3. Methodology

- **Preserved Week-4 Baseline:** `score = 2*(impressions >= 1000) + 2*(staleness_days >= 14) + 1*(position >= 8)` (`score >= 3` binary threshold).
- **Models Evaluated:**
  1. Standardized 4-feature Logistic Regression (`impressions, clicks, staleness_days, position`).
  2. Standardized 7-feature pre-decision Logistic Regression (`log_impressions, log_clicks, staleness_days, position, ctr_pct, days_with_impressions, content_age_days`, `class_weight='balanced'`).
- **Validation Regimes:** 80/20 Stratified Random Split and Client-Grouped Holdout Split (26 train clients / 6 unseen test clients).

In [3]:
print("Week-5 Practice Model Standardized Coefficients (N=100):")
print(pd.DataFrame(M["w05_practice_evaluation"]["coefficients"]).to_string(index=False))

print("\nFlyRank 30k 7-Feature Model Standardized Coefficients (Client-Grouped Holdout):")
print(pd.DataFrame(M["flyrank_30k_evaluation"]["splits"]["client_grouped_holdout"]["logreg_7feat_predecision"]["coefficients"]).to_string(index=False))


Week-5 Practice Model Standardized Coefficients (N=100):
       feature  coefficient  absolute_coefficient
staleness_days     0.640502              0.640502
   impressions     0.432939              0.432939
        clicks     0.113056              0.113056
      position     0.104730              0.104730

FlyRank 30k 7-Feature Model Standardized Coefficients (Client-Grouped Holdout):
              feature  coefficient  absolute_coefficient
      log_impressions     1.236082              1.236082
           log_clicks    -0.957725              0.957725
     content_age_days    -0.399013              0.399013
             position    -0.293176              0.293176
days_with_impressions    -0.210529              0.210529
       staleness_days     0.102167              0.102167
              ctr_pct     0.006194              0.006194


## 4. Results (vs baseline)

In [4]:
w05 = M["w05_practice_evaluation"]
print("=== Population A: W05 Practice Test Split (n=20, Base Rate = 0.6000) ===")
pop_a_df = pd.DataFrame([
    {"Method": "Week-4 Baseline (score >= 3)", **w05["week4_baseline"]},
    {"Method": "Week-5 Logistic Regression (4 feat)", **w05["week5_logistic_regression"]},
])
print(pop_a_df[["Method", "accuracy", "f1", "precision", "recall", "roc_auc", "tp", "fp", "tn", "fn"]].to_string(index=False))

print("\n=== Population B: FlyRank 30k Starter Dataset Across Splits ===")
rows_b = []
for split_name, split_key in [
    ("Stratified 80/20 (n=6,000)", "stratified_random_split"),
    ("Client Holdout (6 clients, n=2,325)", "client_grouped_holdout"),
]:
    sp = M["flyrank_30k_evaluation"]["splits"][split_key]
    for m_key, m_label in [
        ("w04_rule_baseline", "Week-4 Baseline (score >= 3)"),
        ("starter_ref_baseline", "Starter Reference Baseline"),
        ("logreg_4feat_raw", "Logistic Regression (4 raw feat)"),
        ("logreg_7feat_predecision", "Logistic Regression (7 pre-decision feat)"),
    ]:
        d = sp[m_key]
        rows_b.append({
            "Split": split_name,
            "Base_Rate": sp["test_positive_rate"],
            "Method": m_label,
            "Accuracy": d["accuracy"],
            "F1": d["f1"],
            "ROC_AUC": d["roc_auc"],
            "P@20": d["precision_at_20"],
            "P@50": d["precision_at_50"],
        })
print(pd.DataFrame(rows_b).to_string(index=False))


=== Population A: W05 Practice Test Split (n=20, Base Rate = 0.6000) ===
                             Method  accuracy     f1  precision  recall  roc_auc  tp  fp  tn  fn
       Week-4 Baseline (score >= 3)       0.7 0.7857     0.6875  0.9167   0.8542  11   5   3   1
Week-5 Logistic Regression (4 feat)       0.8 0.8571     0.7500  1.0000   0.9375  12   4   4   0

=== Population B: FlyRank 30k Starter Dataset Across Splits ===
                              Split  Base_Rate                                    Method  Accuracy     F1  ROC_AUC  P@20  P@50
         Stratified 80/20 (n=6,000)      0.542              Week-4 Baseline (score >= 3)    0.5432 0.6557   0.5530  0.70  0.74
         Stratified 80/20 (n=6,000)      0.542                Starter Reference Baseline    0.5372 0.5166   0.5787  0.40  0.48
         Stratified 80/20 (n=6,000)      0.542          Logistic Regression (4 raw feat)    0.5498 0.6685   0.5557  0.35  0.46
         Stratified 80/20 (n=6,000)      0.542 Logistic Regress

## 5. Limitations

1. **Observational Associations Only:** High predicted priority indicates historical association with decline, not causal proof that an edit guarantees recovery.
2. **Contemporaneous Proxy Label:** `is_declining_label` is derived from `trend_direction == 'down'` within the 90d window (though all 30d sub-window columns are excluded from $X$).
3. **Cross-Client Base-Rate Shift:** Training clients have a `0.5548` positive rate vs. `0.3910` on the 6 unseen test clients.

In [5]:
print("Exact W05 Test Errors (4 False Positives):")
print(pd.DataFrame(M["w05_practice_evaluation"]["test_errors"])[["test_pos", "item_id", "impressions", "clicks", "staleness_days", "position", "actual", "predicted", "pred_prob"]].to_string(index=False))


Exact W05 Test Errors (4 False Positives):
 test_pos  item_id  impressions  clicks  staleness_days  position  actual  predicted  pred_prob
        5 item_100          904     740              22         2       0          1     0.5336
        9 item_067         1095     460              24         2       0          1     0.5379
       10 item_079          479     476              25        10       0          1     0.5557
       12 item_063         4898     749              19         7       0          1     0.7891


## 6. Ranked recommendations

In [6]:
print("=== Top-10 Refresh Queue: Held-Out Test Clients (n=2,325) ===")
q10 = pd.DataFrame(M["flyrank_30k_evaluation"]["top10_queue_client_holdout"])
print(q10[["rank", "content_id", "priority_score", "pred_prob", "score", "action_full", "is_declining_label"]].to_string(index=False))


=== Top-10 Refresh Queue: Held-Out Test Clients (n=2,325) ===
 rank           content_id  priority_score  pred_prob  score                              action_full  is_declining_label
    1 content_3e79eaafc89d            94.1   0.915925      5 REVIEW_NOW (Refresh & Snippet/CTR Check)                   0
    2 content_8fdbff16a886            93.4   0.905985      5 REVIEW_NOW (Refresh & Snippet/CTR Check)                   0
    3 content_477f7892c1f1            93.0   0.899939      5 REVIEW_NOW (Refresh & Snippet/CTR Check)                   1
    4 content_ef731e95e774            93.0   0.899491      5 REVIEW_NOW (Refresh & Snippet/CTR Check)                   1
    5 content_03582b12af32            92.6   0.894517      5 REVIEW_NOW (Refresh & Snippet/CTR Check)                   1
    6 content_25ebfb5aa399            92.1   0.886778      5 REVIEW_NOW (Refresh & Snippet/CTR Check)                   0
    7 content_c2623272bc49            91.9   0.884947      5 REVIEW_NOW (Refresh & S

## 7. Artifacts the paper embeds

In [7]:
artifacts = [
    REPO_ROOT / "work" / "capstone_report.md",
    REPO_ROOT / "work" / "outputs" / "capstone_metrics.json",
    REPO_ROOT / "docs" / "index.html",
    REPO_ROOT / "submission" / "paper_url.txt",
    REPO_ROOT / "work" / "figures" / "fig1_feature_distributions.svg",
    REPO_ROOT / "work" / "figures" / "fig2_baseline_vs_model_metrics.svg",
    REPO_ROOT / "work" / "figures" / "fig3_logistic_regression_coefficients.svg",
    REPO_ROOT / "work" / "figures" / "fig4_error_analysis_breakdown.svg",
    REPO_ROOT / "work" / "figures" / "fig5_top10_refresh_queue.svg",
]
for a in artifacts:
    assert a.exists(), f"Missing artifact: {a}"
    print(f"[OK] {a.relative_to(REPO_ROOT).as_posix():45s} ({a.stat().st_size:,} bytes)")


[OK] work/capstone_report.md                       (49,348 bytes)
[OK] work/outputs/capstone_metrics.json            (28,740 bytes)
[OK] docs/index.html                               (61,745 bytes)
[OK] submission/paper_url.txt                      (53 bytes)
[OK] work/figures/fig1_feature_distributions.svg   (188,074 bytes)
[OK] work/figures/fig2_baseline_vs_model_metrics.svg (116,620 bytes)
[OK] work/figures/fig3_logistic_regression_coefficients.svg (104,143 bytes)
[OK] work/figures/fig4_error_analysis_breakdown.svg (113,304 bytes)
[OK] work/figures/fig5_top10_refresh_queue.svg     (152,970 bytes)


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`